#BukaVINO

Dalam buku catatan ini, kami akan menunjukkan cara menggunakan toolkit OpenVINO untuk menerapkan model pembelajaran mendalam pada perangkat edge dan mengkuantisasi model untuk mengurangi ukuran model dan latensi inferensi. Kami akan melatih model CNN sederhana pada kumpulan data MNIST, mengonversinya ke format OpenVINO IR, dan mengkuantisasi model tersebut ke presisi INT8. Kami kemudian akan membandingkan ukuran dan kinerja model terkuantisasi dengan model FP32 asli.

## Siapkan OpenVINO

Pertama, kita perlu menginstal OpenVINO, NNCF dan torch

In [1]:
%pip install -q "openvino>=2023.1.0" torch torchvision --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "nncf>=2.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 MB 13.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.9/422.9 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 6.1 MB/s eta 0:00:00


In [2]:
# Mengimpor library PyTorch dan modul-modul terkait untuk deep learning
import torch                # Library utama PyTorch untuk tensor dan operasi jaringan neural
import torch.nn as nn       # Modul PyTorch untuk membangun lapisan jaringan neural
import torch.nn.functional as F  # Berisi fungsi-fungsi aktivasi dan operasi lainnya
import torch.optim as optim  # Modul untuk optimisasi, seperti SGD atau Adam
# Mengimpor torchvision untuk dataset dan transformasi gambar
from torchvision import datasets, transforms  # Untuk memuat dataset dan melakukan preprocessing
# Mengimpor pathlib untuk manipulasi path file secara lintas platform
import pathlib
# Mengimpor numpy untuk operasi numerik
import numpy as np
# Mengimpor OpenVINO untuk inferensi yang dioptimalkan
import openvino as ov  # Library OpenVINO untuk mengoptimalkan model dan menjalankan inferensi
# Mengimpor NNCF (Neural Network Compression Framework) untuk kompresi model
import nncf  # Framework untuk mengaplikasikan teknik kompresi seperti quantization dan pruning


INFO:nncf:NNCF initialized successfully. Supported frameworks detected: torch, tensorflow, openvino


## Model Kereta Api

Selanjutnya, tentukan dan latih model CNN sederhana pada kumpulan data MNIST

In [3]:
# Transformasi data: mengubah gambar ke tensor dan menormalisasi
transform = transforms.Compose([
    transforms.ToTensor(),  # Mengubah gambar menjadi tensor
    transforms.Normalize((0.1307,), (0.3081,))  # Normalisasi dengan mean=0.1307 dan std=0.3081
])

# Memuat dataset MNIST untuk pelatihan dan pengujian
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)  # Dataset pelatihan
test_dataset = datasets.MNIST('./data', train=False, transform=transform)  # Dataset pengujian

# Mendefinisikan arsitektur jaringan neural sederhana
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=12, kernel_size=3)  # Lapisan konvolusi
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # Lapisan pooling
        self.fc = nn.Linear(12 * 13 * 13, 10)  # Lapisan fully connected (linear)

    def forward(self, x):
        x = x.view(-1, 1, 28, 28)  # Mengubah input menjadi ukuran (batch_size, 1, 28, 28)
        x = F.relu(self.conv1(x))  # Aktivasi ReLU setelah lapisan konvolusi
        x = self.pool(x)  # Pooling untuk mengurangi dimensi spasial
        x = x.view(x.size(0), -1)  # Flatten tensor sebelum masuk ke lapisan fully connected
        x = self.fc(x)  # Lapisan fully connected
        output = F.log_softmax(x, dim=1)  # Fungsi softmax log untuk output probabilitas log
        return output

# Membuat DataLoader untuk memuat data dalam batch
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32)  # DataLoader pelatihan
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32)  # DataLoader pengujian

# Menentukan perangkat (CPU)
device = "cpu"

# Jumlah epoch untuk pelatihan
epochs = 1

# Membuat model dan memindahkannya ke perangkat
model = Net().to(device)

# Optimizer menggunakan Adam
optimizer = optim.Adam(model.parameters())

# Mode pelatihan untuk model
model.train()

# Loop pelatihan
for epoch in range(1, epochs + 1):
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)  # Memindahkan data ke perangkat
        optimizer.zero_grad()  # Mengatur gradien ke nol
        output = model(data)  # Melakukan forward pass
        loss = F.nll_loss(output, target)  # Menghitung loss (Negative Log-Likelihood)
        loss.backward()  # Backpropagation
        optimizer.step()  # Memperbarui parameter model
        print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
            epoch, batch_idx * len(data), len(train_loader.dataset),
            100. * batch_idx / len(train_loader), loss.item()))

# Menyimpan model ke direktori `./models`
MODEL_DIR = pathlib.Path("./models")
MODEL_DIR.mkdir(exist_ok=True)  # Membuat direktori jika belum ada
torch.save(model.state_dict(), MODEL_DIR / "original_model.p")  # Menyimpan bobot model


Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 9.91M/9.91M [00:00<00:00, 15.9MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 28.9k/28.9k [00:00<00:00, 499kB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 1.65M/1.65M [00:00<00:00, 4.35MB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 4.54k/4.54k [00:00<00:00, 3.51MB/s]


Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.365148
Train Epoch: 1 [32/60000 (0%)]	Loss: 2.288796
Train Epoch: 1 [64/60000 (0%)]	Loss: 2.128072
Train Epoch: 1 [96/60000 (0%)]	Loss: 1.932519
Train Epoch: 1 [128/60000 (0%)]	Loss: 2.018445
Train Epoch: 1 [160/60000 (0%)]	Loss: 2.027848
Train Epoch: 1 [192/60000 (0%)]	Loss: 1.444902
Train Epoch: 1 [224/60000 (0%)]	Loss: 1.683165
Train Epoch: 1 [256/60000 (0%)]	Loss: 1.679698
Train Epoch: 1 [288/60000 (0%)]	Loss: 1.257255
Train Epoch: 1 [320/60000 (1%)]	Loss: 1.346398
Train Epoch: 1 [352/60000 (1%)]	Loss: 1.146433
Train Epoch: 1 [384/60000 (1%)]	Loss: 1.201220
Train Epoch: 1 [416/60000 (1%)]	Loss: 1.000956
Train Epoch: 1 [448/60000 (1%)]	Loss: 0.969720
Train Epoch: 1 [480/60000 (1%)]	Loss: 1.474514
Train Epoch: 1 [512/60000 (1%)]	Loss: 1.289210
Train Epoch: 1 [544/60000 (1%)]	Loss: 0.857666
Train Epoch: 1 [576/60000 (1%)]	Loss: 1.245960
Train Epoch: 1 [608/60000 (1%)]	Loss:

## Konversikan ke OpenVINO IR

Kemudian, konversikan model ke format OpenVINO IR

In [4]:
# Menginisialisasi OpenVINO Core untuk mengelola model dan perangkat inferensi
core = ov.Core()

# Mendapatkan contoh input dari test_loader
example_input = next(iter(test_loader))[0]  # Mengambil batch pertama dari DataLoader (hanya gambar)

# Mengonversi model PyTorch ke format OpenVINO IR (Intermediate Representation)
ov_model = ov.convert_model(model, example_input=example_input)

# Menyimpan model dalam format OpenVINO IR dengan ekstensi .xml
ov.save_model(ov_model, MODEL_DIR / f"openvino_ir.xml")


No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


## Kuantisasi

Untuk mengkuantisasi model menggunakan NNCF, pertama-tama, buat fungsi transformasi untuk mengonversi tensor obor ke array NumPy, lalu gunakan fungsi yang dibuat bersama dengan pemuat data pytorch untuk membuat kumpulan data kalibrasi menggunakan kelas `Dataset` dari NNCF. Selanjutnya, kuantisasi model menggunakan fungsi `quantize` dari NNCF. Terakhir, kompilasi model terkuantisasi dan simpan sebagai format OpenVINO IR.

In [5]:
# Fungsi transformasi untuk dataset kalibrasi
def transform_fn(data_item):
    images, _ = data_item  # Mengambil gambar dan mengabaikan label
    return images.numpy()  # Mengubah tensor PyTorch menjadi array NumPy

# Membuat dataset kalibrasi menggunakan NNCF
# Dataset ini akan digunakan untuk proses quantization
calibration_dataset = nncf.Dataset(train_loader, transform_fn)

# Melakukan quantization pada model OpenVINO menggunakan dataset kalibrasi
quantized_model = nncf.quantize(ov_model, calibration_dataset)

# Mengompilasi model quantized untuk inferensi
model_int8 = ov.compile_model(quantized_model)

# Mengambil input FP32 dari test_loader untuk melakukan pengujian inferensi
input_fp32 = next(iter(test_loader))[0][0:1]  # Mengambil satu gambar pertama dari batch

# Melakukan inferensi pada model INT8
res = model_int8(input_fp32)

# Menyimpan model yang telah diquantisasi dalam format OpenVINO IR
ov.save_model(quantized_model, MODEL_DIR / f"quant_openvino_ir.xml")


Output()

Output()

## Periksa Ukuran

Bandingkan ukuran model FP32 dan INT8

In [6]:
# Menampilkan daftar file dalam direktori MODEL_DIR
%ls -lh {MODEL_DIR}

total 176K
-rw-r--r-- 1 root root 40K Jan  3 15:22 openvino_ir.bin
-rw-r--r-- 1 root root 11K Jan  3 15:22 openvino_ir.xml
-rw-r--r-- 1 root root 82K Jan  3 15:22 original_model.p
-rw-r--r-- 1 root root 21K Jan  3 15:22 quant_openvino_ir.bin
-rw-r--r-- 1 root root 16K Jan  3 15:22 quant_openvino_ir.xml


## Periksa Akurasi

Evaluasi keakuratan model INT8 dan bandingkan dengan model FP32

In [7]:
# Fungsi untuk menguji model OpenVINO pada dataset pengujian
def test_ov(model, data_loader):
    # Mengompilasi model OpenVINO untuk perangkat target (misalnya, CPU)
    compiled_model = ov.compile_model(model)

    test_loss = 0  # Inisialisasi total loss
    correct = 0    # Inisialisasi jumlah prediksi yang benar

    # Iterasi melalui data loader
    for data, target in data_loader:
        # Melakukan inferensi dengan model OpenVINO
        output = torch.tensor(compiled_model(data)[0])  # Konversi output OpenVINO ke tensor PyTorch

        # Menghitung loss batch dan menjumlahkan
        test_loss += F.nll_loss(output, target, reduction='sum').item()

        # Mendapatkan prediksi (indeks dengan probabilitas tertinggi)
        pred = output.argmax(dim=1, keepdim=True)

        # Menjumlahkan prediksi yang benar
        correct += pred.eq(target.view_as(pred)).sum().item()

    # Menghitung rata-rata loss
    test_loss /= len(data_loader.dataset)

    # Menghitung akurasi
    return 100. * correct / len(data_loader.dataset)

# Menguji model asli (FP32)
acc = test_ov(ov_model, test_loader)
print(f"Accuracy of original model: {acc}")

# Menguji model yang telah diquantisasi (INT8)
qacc = test_ov(quantized_model, test_loader)
print(f"Accuracy of quantized model: {qacc}")


Accuracy of original model: 96.55
Accuracy of quantized model: 96.65
